# Upload Wikipedia Popularity Dataset to HuggingFace

This notebook uploads the Wikipedia pageviews dataset to HuggingFace Hub.

**Prerequisites:** 
- HuggingFace account and WRITE token in `.env` as `HUGGINGFACE_TOKEN`
- Dataset file at `data/popularity_table_YEAR_avg.parquet`

## Setup: Authentication and Configuration

Load environment variables, authenticate with HuggingFace, and set configuration.

In [12]:
import sys

sys.path.append("..")

import os
from dotenv import load_dotenv

from config import DATA_DIR

load_dotenv()

# Configuration
YEAR = 2020
DATASET_PATH = DATA_DIR / f"popularity_table_{YEAR}_avg.parquet"
HUGGINGFACE_REPO_ID = f"Cyro1/enwiki_pageviews_{YEAR}_m"
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

# README content for HuggingFace dataset card
README_TEXT = f"""---
license: mit
task_categories:
  - other
  - text-classification
language:
  - en
size_categories:
  - 1M<n<10M
tags:
  - wikipedia
  - pageviews
  - popularity
  - knowledge-graph
  - bias-detection
---

# English Wikipedia Pageviews {YEAR} (Monthly Average)

This dataset links Wikipedia article IDs (from the 1st September 2019 Wikipedia dump) to their average monthly pageviews recorded during {YEAR}.

## Features

| Column | Type | Description |
|--------|------|-------------|
| `wikipedia_id` | int64 | Wikipedia article ID (page_id) |
| `wikipedia_title` | string | Wikipedia article title |
| `popularity_avg` | float64 | Average monthly pageviews across {YEAR} |
| `rank_avg` | float64 | Average rank of the article |

## Stats

- **Articles**: 6.4M Wikipedia articles
- **Source**: Wikimedia pageview dumps for {YEAR}
- **Split**: 90% train / 10% test (seed=42)

## Usage

```python
from datasets import load_dataset

ds = load_dataset("Cyro1/enwiki_pageviews_2021_m")
df = ds["train"].to_pandas()

# Merge with your dataset
merged = your_df.merge(
    df[["wikipedia_id", "monthly_avg_pageviews"]], 
    left_on="document_id", 
    right_on="wikipedia_id"
)
```
"""

In [13]:
from huggingface_hub import login

print(f"HF_TOKEN is set: {HF_TOKEN is not None}")
login(token=HF_TOKEN)

HF_TOKEN is set: True


## Load Dataset

Load the parquet file and rename `id` column to `wikipedia_id` for clarity.

In [14]:
import pandas as pd
import datasets

df = pd.read_parquet(DATASET_PATH)

print(f"Loaded {len(df):,} articles")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 5,903,530 articles
Columns: ['wikipedia_id', 'wikipedia_title', 'popularity_avg', 'rank_avg']


,wikipedia_id,wikipedia_title,popularity_avg,rank_avg
0,1000,Hercule Poirot,65156.333333,1.314608e+04
1,10000,Eiffel,841.416667,8.789471e+05
2,10000001,Juan que reía,22.000000,4.463454e+06
3,10000009,La Noche del hurto,30.500000,4.022241e+06
4,1000001,NPU,1439.333333,6.024527e+05


## Upload to HuggingFace

Convert to HuggingFace Dataset, create train/test split, and push to hub with README.

In [15]:
from huggingface_hub import HfApi
import datasets

# Authenticate only when uploading (lazy import)


# Reset index and convert to HuggingFace dataset
# preserve_index=False ensures wikipedia_id is kept as a feature column
df_reset = df.reset_index(drop=True)
dataset = datasets.Dataset.from_pandas(df_reset, preserve_index=False)

# Create train/test split
dataset_dict = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(dataset_dict['train']):,} | Test: {len(dataset_dict['test']):,}")

# Upload dataset
print(f"\nUploading to {HUGGINGFACE_REPO_ID}...")
dataset_dict.push_to_hub(HUGGINGFACE_REPO_ID)

# Upload README
print("Uploading README...")
api = HfApi()
api.upload_file(
    path_or_fileobj=README_TEXT.encode(),
    path_in_repo="README.md",
    repo_id=HUGGINGFACE_REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN,
)

print(f"\n✓ Upload complete!")
print(f"📊 https://huggingface.co/datasets/{HUGGINGFACE_REPO_ID}")

Train: 5,313,177 | Test: 590,353

Uploading to Cyro1/enwiki_pageviews_2020_m...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5314 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/591 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Uploading README...

✓ Upload complete!
📊 https://huggingface.co/datasets/Cyro1/enwiki_pageviews_2020_m
